# Create Text Embedding Models

## What is Contrastive Learning?
- Contrastive learning is a technique that aims to train an embedding model such that similar documents are closer in vector space while dissimilar documents are further apart.
- The underlying idea of contrastive learning is that the best way to learn and model similarity/dissimilarity between documents is by feeding a model examples of similar and dissimilar pairs.
- In order to accurately capture the semantic nature of a document, it often needs to be contrasted with another document for a model to learn what makes it different or similar
- This contrasting procedure is quite powerful and relates to the context in which documents are written. This high-level procedure is demonstrated in
- There are many ways we can apply contrastive learning to create text embedding models but the most well-known technique and framework is sentence-transformers.

## SBERT
 - Cross-encoder allows two sentences to be passed to the transformer network simultaneously to predict the extent to which the two sentences are similar.
 - It does so by adding a classification head to the original architecture that can output a similarity score. 
 - However, the number of computations rises quickly when you want to find the highest pair in a collection of 10,000 sentences. That would require n·(n−1)/2 = 49,995,000 inference computations and therefore generates significant overhead.
 -  Moreover, a cross-encoder generally does not generate embeddings, as shown in img6. Instead, it outputs a similarity score between the input sentences.

 - Instead, the authors of sentence-transformers approached the problem differently and searched for a method that is fast and creates embeddings that can be compared semantically. 
   - The result is an elegant alternative to the original cross-encoder architecture. 
   - Unlike a cross-encoder, in sentence-transformers the classification head is dropped, and instead mean pooling is used on the final output layer to generate an embedding. 
   - This pooling layer averages the word embeddings and gives back a fixed dimensional output vector. This ensures a fixed-size embedding.

## Crating an embedding model
### Generating Contrastive Examples
- When pretraining your embedding model, you will often see data being used from natural language inference (NLI) datasets. 
- NLI refers to the task of investigating whether, for a given premise, it entails the hypothesis (entailment), contradicts it (contradiction), or neither (neutral).

- In our Example, we'll use GLUE dataset that consistes of 9 language understanding tasks to evaluate an anlyze the model performance.
- One of these tasks is the Multi-Genre Natural Language Inference (MNLI) corpus, which is a collection of 392,702 sentence pairs annotated with entailment (contradiction, neutral, entailment). 
- We will be using a subset of the data, 50,000 annotated sentence pairs, to create a minimal example that does not need to be trained for hours on end.


In [1]:
pip install -q accelerate>=0.27.2 peft>=0.9.0 bitsandbytes>=0.43.0 transformers>=4.38.2 trl>=0.7.11 sentencepiece>=0.1.99

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -q sentence-transformers>=3.0.0 mteb>=1.1.2 datasets>=2.18.0

Note: you may need to restart the kernel to use updated packages.


### Data

In [1]:
from datasets import load_dataset
# Load MNLI dataset from GLUE
# 0=entailment, 1=neutral, 2=contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_dataset[0]

{'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.',
 'hypothesis': 'Product and geography are what make cream skimming work. ',
 'label': 1}

### Train the model
- Now that we have our dataset with training examples, we will need to create our embedding model.
- We typically choose an existing sentence-transformers model and fine-tune that model, but in this example, we are going to train an embedding from scratch.
- This means that we will have to define two things.
  -  **First, a pretrained Transformer model that serves as embedding individual words.**
    - We will use the BERT base model (uncased) as it is a great introduction model.
    - However, many others exist that also have been evaluated using sentence-transformers.  Most notably, microsoft/mpnet-base often gives good results when used as a word embedding model: https://www.sbert.net/docs/sentence_transformer/training_overview.html#best-transformer-model
  

In [2]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("bert-base-uncased")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3618.22it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


- Next, we will need to define a loss function over which we will optimize the model. 
### Loss function


In [3]:
from sentence_transformers import losses

# Define the loss function. In soft-max loss, we will also need to explicitly set the number of labels.
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3
)

C:\Users\pc\AppData\Local\Temp\ipykernel_29608\1944009623.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses
C:\Users\pc\AppData\Local\Temp\ipykernel_29608\1944009623.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
The `sentence_embedding_dimension` argument was renamed and is now deprecated. Please use `embedding_dimension` instead.


- Before we train our model, we define an evaluator to evaluate the model’s performance during training, which also determines the best model to save.
- We can perform evaluation of the performance of our model using the Semantic Textual Similarity Benchmark (STSB).
  - It is a collection of human-labeled sentence pairs, with similarity scores between 1 and 5.

### Evaluation


In [4]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for steb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine",
)

C:\Users\pc\AppData\Local\Temp\ipykernel_29608\2459235318.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


### Training


In [5]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define training arguments
args = SentenceTransformerTrainingArguments(
    output_dir='output/base_embedding_model',
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100
)


C:\Users\pc\AppData\Local\Temp\ipykernel_29608\1133874191.py:1: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


- num_train_epochs: The number of training rounds. We keep this at 1 for faster training but it is generally advised to increase this value.
- per_device_train_batch_size: The number of samples to process simultaneously on each device (e.g., GPU or CPU) during evaluation. Higher values generally means faster training.
- per_device_eval_batch_size: The number of samples to process simultaneously on each device (e.g., GPU or CPU) during evaluation. Higher values generally means faster evaluation.
- warmup_steps: The number of steps during which the learning rate will be linearly increased from zero to the initial learning rate defined for the training process. Note that we did not specify a custom learning rate for this training process.
- fp16: By enabling this parameter we allow for mixed precision training, where computations are performed using 16-bit floating-point numbers (FP16) instead of the default 32-bit (FP32). This reduces memory usage and potentially increases the training speed.

In [6]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

C:\Users\pc\AppData\Local\Temp\ipykernel_29608\2767052707.py:1: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import SentenceTransformerTrainer
Column 'hypothesis' is at index 1, whereas a column with this name is usually expected at index 0. Note that the column order can be important for some losses, e.g. MultipleNegativesRankingLoss will always consider the first column as the anchor and the second as the positive, regardless of the dataset column names. Consider renaming the columns to match the expected order, e.g.:
dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.069216
200,0.951879
300,0.892270
400,0.850261
500,0.822246
600,0.831090
700,0.807341
800,0.795445
900,0.771121
1000,0.762719


Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.70s/it]


TrainOutput(global_step=1563, training_loss=0.8135789418479836, metrics={'train_runtime': 420.7317, 'train_samples_per_second': 118.841, 'train_steps_per_second': 3.715, 'total_flos': 0.0, 'train_loss': 0.8135789418479836, 'epoch': 1.0})

 After training our model, we can use the evaluator to get the perfromance on this single tsk:
 

In [7]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.531152024044593, 'spearman_cosine': 0.6134434108883291}

- We get several different distance measures. The one we are interested in most is 'pearson_cosine', which is the cosine similarity between centered vectors. 
- It is a value between 0 and 1 where a higher value indicates higher degrees of similarity. We get a value of 0.53, which we consider a baseline throughout this chapter.


### Tip
Larger batch sizes tend to work better with multiple negative rankings (MNR) loss as a larger batch makes the task more difficult. The reason for this is that the model needs to find the best matching sentence from a larger set of potential pairs of sentences. You can adapt the code to try out different batch sizes and get a feeling of its effects.

## In-Depth Evaluation
- A good embedding model is more than just a good score on the STSB benchmark!
- The GLUE benchmark has a number of tasks for which we can evaluate our embedding model.
- However, there exist many more benchmarks that allow for the evaluation of embedding models
  - To unify this evaluation procedure, the massive Text Embedding Benchmark (MTEB) was developed
  - MTEB spans 8 embedding tasks that cover 58 datasets and 112 languages.

- Link: https://huggingface.co/spaces/mteb/leaderboard
- Link: https://pypi.org/project/mteb/

In [9]:
import mteb

# Choose evaluation task
# Choose evaluation task
evaluation = mteb.get_tasks(tasks=["Banking77Classification.v2"])

# Calculate results
results = mteb.evaluate(embedding_model, tasks=evaluation)
results

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\mteb\models\model_meta.py:753: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\mteb\models\model_meta.py:715: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),
Evaluating task Banking77Classification.v2:   0%|          | 0/1 [00:00<?, ?it/s]d:\2026-courses\LLMs-Handson\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\datasets--mteb--banking77. Caching files will still work but in a degraded version that might require more space on your disk. This warning can

ModelResult(model_name='SentenceTransformer based on google-bert/bert-base-uncased', model_revision='86b5e0934494bd15c9632b12f734a8a67f723594', task_results=[...](#1), ...)

In [13]:
results

ModelResult(model_name='SentenceTransformer based on google-bert/bert-base-uncased', model_revision='86b5e0934494bd15c9632b12f734a8a67f723594', task_results=[...](#1), ...)

In [28]:
# Inspect the full results structure
print("Model Name:", results.model_name)
print("\nModel Revision:", results.model_revision)
print("\nTask Results:")
for task_result in results.task_results:
    print(f"\n  Task: {task_result.task_name}")
    print(f"  Scores: {task_result.scores['test'][0]['scores_per_experiment']}")
    if hasattr(task_result, 'evaluation_time'):
        print(f"  Evaluation Time: {task_result.evaluation_time}")
    if hasattr(task_result, 'main_score'):
        print(f"  Main Score: {task_result.main_score}")

Model Name: SentenceTransformer based on google-bert/bert-base-uncased

Model Revision: 86b5e0934494bd15c9632b12f734a8a67f723594

Task Results:

  Task: Banking77Classification.v2
  Scores: [{'accuracy': 0.5744473342002601, 'f1': 0.5718957161277722, 'f1_weighted': 0.5718924851449991, 'precision': 0.5822662270049382, 'precision_weighted': 0.5822825784119708, 'recall': 0.5744588744588744, 'recall_weighted': 0.5744473342002601, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.6007802340702211, 'f1': 0.5994701248522412, 'f1_weighted': 0.599596883805227, 'precision': 0.613377705305322, 'precision_weighted': 0.6135057675394631, 'recall': 0.6006410256410256, 'recall_weighted': 0.6007802340702211, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.5842002600780234, 'f1': 0.5809730978162526, 'f1_weighted': 0.5811132039824065, 'precision': 0.5936165854030808, 'precision_weighted': 0.5937433249859798, 'recall': 0.5840659340659341, 'recall_weighted': 0.5842002600780234, 'ap': None, 'ap_weighted': No

### Tip

Whenever you are done training and evaluating your model, it is important to restart the notebook. This will clear your VRAM up for the next training examples throughout this chapter. By restarting the notebook, we can be sure that all VRAM is cleared.

 ## VRAM Clean-up - You will need to run the code below to partially empty the VRAM (GPU RAM). If that does not work, it is advised to restart the notebook instead. You can check the resources on the right-hand side (if you are using Google Colab) to check whether the used VRAM is indeed low. You can also run !nivia-smi to check current usage.

In [ ]:
# # Empty and delete trainer/model
# trainer.accelerator.clear()
# del trainer, embedding_model

# # Garbage collection and empty cache
# import gc
# import torch

# gc.collect()
# torch.cuda.empty_cache()



In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## Loss Functions
- We trained our model using softmax loss to illustrate how one of the first sentence-transformers models was trained.
- However, not only is there a large variety of loss functions to choose from, but softmax loss is generally not advised as there are more: https://www.sbert.net/docs/package_reference/sentence_transformer/losses.html

- Instead of going through every single loss function out there, there are two loss functions that are typically used and seem to perform generally well, namely:
 - Cosine similarity
 - Multiple negatives ranking (MNR) loss